In [1]:
import os
import glob

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_21792\2162706441.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
# ---------------------------------------------------------------------------
# 설정값
# ---------------------------------------------------------------------------
DATA_DIR = "../rag_data"
DB_DIR = "../chroma_govfund_db"
COLLECTION_NAME = "govfund_guide"
EMBEDDING_MODEL = "bge-m3"  # 사용자 환경에 맞게 변경 (ollama pull bge-m3 필요)
CHUNK_SIZE = 400
CHUNK_OVERLAP = 50

# 파일명 -> 카테고리 매핑 (post_processing / routing 에서 함께 사용)
CATEGORY_MAP = {
    "(공고문)_2026년도_중앙부처_및_지자체_창업지원사업_통합공고문(제2025-648호,_2025.12.19.).pdf": "notice",
}


In [7]:
# ---------------------------------------------------------------------------
# 1. pdf 문서 로딩 -> Document 생성
# ---------------------------------------------------------------------------

documents: list[Document] = []
pdf_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.pdf")))

if not pdf_files:
    raise FileNotFoundError(
        f"'{DATA_DIR}' 폴더에서 pdf 파일을 찾을 수 없습니다. "
        "rag_data 폴더 위치를 확인하세요."
    )

for path in pdf_files:
    filename = os.path.basename(path)
    with open(path, "r", encoding="utf-8") as f:
        loader_notice = PyPDFLoader(path)

        pdf_docs = loader_notice.load()

        category = CATEGORY_MAP.get(filename, "general")

        for doc in pdf_docs:
            doc.metadata["category"] = category

        documents.extend(pdf_docs)

        total_chars = sum(len(doc.page_content) for doc in pdf_docs)

        print(f"  - 로드 완료: {filename} ({total_chars:,}자, category={category})")


  - 로드 완료: (공고문)_2026년도_중앙부처_및_지자체_창업지원사업_통합공고문(제2025-648호,_2025.12.19.).pdf (92,234자, category=notice)


In [9]:
# ---------------------------------------------------------------------------
# 2. Chunk 분할
# ---------------------------------------------------------------------------

print("all_docs:",len(documents))

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE,chunk_overlap=CHUNK_OVERLAP)

chunks = splitter.split_documents(documents)

# chunk 순번 metadata 부여 (디버깅/추적용)
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_index"] = i

print(len(chunks))


all_docs: 105
345


In [10]:
# ---------------------------------------------------------------------------
# 3~4. Embedding 생성 + Chroma Vector DB 저장
# ---------------------------------------------------------------------------

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL,base_url="http://10.8.0.1:11434")

if os.path.exists(DB_DIR):
    print(f"\n기존 '{DB_DIR}' 폴더가 존재합니다. 동일 컬렉션을 다시 생성/덮어씁니다.")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=DB_DIR,
)



